# RAG on Azure — Day 8: Hybrid Search (Keyword + Vector) 

**Use case:** technical documentation Q&A — API references, error codes, SKUs, version numbers, CLI flags. Exactly the kind of content where pure vector search confidently returns the *wrong* exact term, and pure keyword search misses the meaning.

- A **dual-mode index**: same fields as before, but the schema marks `content` as searchable (BM25-indexed) *and* keeps the vector field.
- Three retrieval modes side by side: **pure vector**, **pure keyword (BM25)**, and **hybrid (RRF fusion)**.
- Azure AI Search applies **Reciprocal Rank Fusion** automatically when both `search_text` and `vector_queries` are passed — no custom merging code.
- A demo that contrasts where each mode wins, then shows hybrid winning on both kinds of question.

## 0. Install dependencies

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import openai, httpx
print("openai :", openai.__version__)
print("httpx  :", httpx.__version__)

openai : 1.55.3
httpx  : 0.28.1


## 1. Setup — three lines, thanks to `common/`

In [ ]:
import sys
sys.path.insert(0, "..")

from common import load_config, get_openai_client, get_search_client, get_search_index_client
from common import chunk_text, load_pdfs, extract_sections
from common import embed_texts, retrieve, retrieve_keyword, retrieve_hybrid
from common import answer

config = load_config(Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env"))
openai = get_openai_client(config)
index_client = get_search_index_client(config)
INDEX_NAME = "rag-techdocs-day8"
print("Setup OK. Index:", INDEX_NAME)

ImportError: cannot import name 'retrieve_keyword' from 'common' (c:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\day-08-hybrid-search\..\common\__init__.py)

## 2. Load and chunk the technical PDFs

The same section-aware loader from Day 6, pointed at the new technical-docs corpus.

In [ ]:
from pathlib import Path
import re

DATA_DIR = Path("../data-day8")
print("Looking in:", DATA_DIR.resolve())
raw_docs = load_pdfs(DATA_DIR)
assert raw_docs, f"No PDFs in {DATA_DIR.resolve()}"

records = []
for d in raw_docs:
    sections = extract_sections(d["text"]) or [("Body", d["text"])]
    for section_title, section_body in sections:
        safe_src = re.sub(r"[^A-Za-z0-9_-]", "_", d["source"])
        safe_sec = re.sub(r"[^A-Za-z0-9_-]", "_", section_title)[:30]
        for i, chunk in enumerate(chunk_text(section_body)):
            records.append({
                "id":      f"{safe_src}-{safe_sec}-{i}",
                "content": chunk,
                "source":  d["source"],
                "section": section_title,
            })

print(f"\n{len(records)} chunks from {len(raw_docs)} PDFs.")
print("Example chunk:")
print(records[0]["content"][:160], "...")

## 3. Embed

In [ ]:
chunk_vectors = embed_texts(openai, [r["content"] for r in records], config["embedding_deployment"])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} chunks. Dim: {EMBED_DIM}")

## 4. Create the dual-mode index

The key detail: `content` is a **`SearchableField`**, which means Azure AI Search builds a BM25 keyword index over it automatically. The vector field stays as before. Same schema — but now we can search it three ways.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),     # <-- BM25 indexed
    SimpleField(name="source",  type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="section", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vs = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index (ok):", type(e).__name__)

index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vs))
print(f"Index '{INDEX_NAME}' created with both BM25 + vector capabilities.")

## 5. Upload

In [ ]:
search = get_search_client(config, INDEX_NAME)
docs = [{**r, "contentVector": v} for r, v in zip(records, chunk_vectors)]
for i in range(0, len(docs), 100):
    search.upload_documents(documents=docs[i:i+100])
print(f"Uploaded {len(docs)} chunks.")

## 6. The three retrieval modes — head to head

A small helper that runs all three modes on the same query and prints the top hits side by side.

In [ ]:
SELECT = ["content", "source", "section"]

def compare_modes(query, k=4):
    print(f"\nQuery: {query!r}")
    print("=" * 80)

    vec = retrieve(search, openai, query, config["embedding_deployment"], k=k, select=SELECT)
    kw  = retrieve_keyword(search, query, k=k, select=SELECT)
    hyb = retrieve_hybrid(search, openai, query, config["embedding_deployment"], k=k, select=SELECT)

    for label, hits in [("VECTOR ONLY", vec), ("KEYWORD ONLY", kw), ("HYBRID (RRF)", hyb)]:
        print(f"\n  -- {label} --")
        if not hits:
            print("     (no results)")
            continue
        for h in hits:
            preview = (h["content"][:90] + "...") if len(h["content"]) > 90 else h["content"]
            print(f"     [{h['score']:.3f}] {h['source']} - {h['section']}")
            print(f"            {preview}")

## 7. Demo A — exact code match: where pure vector struggles

`ERR-4012` is a specific error code. Vector embeddings tend to think all `ERR-XXXX` codes look semantically similar, so pure vector may rank the wrong code's passage above the right one. Keyword search nails it instantly. Hybrid keeps the precision *and* still allows semantic matches.

In [ ]:
compare_modes("What does error ERR-4012 mean?")

## 8. Demo B — exact SKU lookup: pure keyword's natural home

A SKU code like `AC-CORE-PRO-MONTHLY` is a token, not a concept. Vector search can wander to the wrong SKU's row. Keyword and hybrid lock onto the exact row.

In [ ]:
compare_modes("What is the price of AC-CORE-PRO-MONTHLY?")

## 9. Demo C — conceptual query: pure keyword's blind spot

Now the opposite case. "How do I cancel a subscription?" never says the word "cancel" — the docs use "DELETE /v2/subscriptions" and "subscription cancel". Pure keyword has nothing to match; vector finds the right passage. Hybrid still wins because it gets BOTH signals.

In [ ]:
compare_modes("How do I cancel a subscription via the API?")

## 10. Demo D — version number: a token vector hates

`v3.7.2` is a string identifier with no semantic content. Vector search drifts to the wrong release notes. Hybrid pins it.

In [ ]:
compare_modes("What changed in v3.7.2?")

## 11. End-to-end answer using hybrid

Generation is unchanged — we just route retrieval through `retrieve_hybrid` instead of `retrieve`. Same `answer()` from `common/`.

In [ ]:
def hybrid_answer(question, k=5):
    hits = retrieve_hybrid(search, openai, question, config["embedding_deployment"], k=k, select=SELECT)
    return answer(openai, config["chat_deployment"], question, hits, cite=True), hits

for q in [
    "What does error ERR-4012 mean?",
    "What's the price of AC-CORE-PRO-MONTHLY?",
    "How do I cancel a subscription via the API?",
    "What security issue was fixed in v3.7.2?",
]:
    ans, hits = hybrid_answer(q)
    print(f"\nQ: {q}")
    print(f"A: {ans}")
    print(f"Sources: {sorted({h['source'] for h in hits})}")
    print("-" * 78)

## ✅ Day 8 checklist

- [ ] Index has both BM25 (`SearchableField`) and vector fields on `content`
- [ ] `retrieve_keyword` returns BM25 results, no embedding call
- [ ] `retrieve_hybrid` returns fused (RRF) results, single API call
- [ ] On the `ERR-4012` query, pure vector misses the right chunk and hybrid finds it
- [ ] On the conceptual "cancel a subscription" query, pure keyword misses and hybrid still wins
- [ ] The end-to-end `hybrid_answer` returns correct answers with citations

### What you learned today
- **Vector ≠ better.** Vector is one shape of relevance (semantic). BM25 is another (exact-term). They complement, not replace.
- **Hybrid is free in Azure AI Search.** Passing both `search_text` and `vector_queries` triggers RRF — no extra service, no custom code.
- **Hybrid is the new default for any corpus with codes, SKUs, version strings, function names, or jargon.** Which is most real-world business content.

### Next up — Day 9: reranking
Pull a large candidate set (top 30–50) and reorder it with Azure AI Search's **semantic ranker** (a cross-encoder under the hood). The next-biggest precision boost, layered on top of today's hybrid retrieval.